In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, asc, to_date, to_timestamp, when, now, lit, avg, current_date, coalesce, trim, regexp_replace
from pyspark.sql.types import IntegerType, DoubleType, StringType, BooleanType
import os

spark = SparkSession.builder \
    .appName("Netflix_transform") \
    .master("local[*]") \
    .getOrCreate()


df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", '"') \
    .csv("/app/netflix_titles.csv")

# Step 1: convert mixed format
clean_df = df.withColumn(
    "date_added",
    coalesce(
        to_date(col("date_added"), "MMMM d, yyyy"),
        to_date(col("date_added"), "yyyy-MM-dd")
    )
)

# Step 2: fill null
clean_df = clean_df.withColumn(
    "date_added",
    when(col("date_added").isNull(), current_date())
    .otherwise(col("date_added"))
)

clean_df = clean_df.fillna({'rating':'UR'})

clean_df = clean_df.withColumn('rating',
                              when(col('rating').rlike("min"),'none')
                              .otherwise(col('rating')))

clean_df = clean_df.withColumn('duration',
                              when(col('duration').isNull(),'0 min')
                                  .otherwise(col('duration')))
clean_df = clean_df.withColumn('duration',
                   when(col('duration').rlike("Seasons|Season"), '100+ min')
                   .otherwise(col('duration')))

clean_df = clean_df.withColumn('listed_in',trim(col('listed_in')))

clean_df = clean_df.withColumn('country',
                              when(col('country').isNull(),'Unknown')
                              .otherwise(col('country')))
clean_df = clean_df.withColumn('cast',
                              when(col('cast').isNull(),'Unknown cast')
                              .otherwise(col('cast')))
clean_df = clean_df.withColumn('director',
                              when(col('director').isNull(),'Unknown director')
                              .otherwise(col('director')))

# Step 3: check nulls
clean_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in clean_df.columns
]).show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 14:01:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   0|    0|       0|   0|      0|         0|           0|     0|       0|        0|          0|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [ ]:
# Data Analysis

## Step 1: Top Directors by Number of Movies

TD = clean_df.groupBy('director').count().orderBy(desc('count'))
TD.filter(col('director') != 'Unknown director').show(10)

## Step 2: Most frequent cast in movies/series

FC = clean_df.groupBy('cast').count().orderBy(desc('count'))
FC.filter(col('cast') != 'Unknown cast').show(10)

## Step 3: Group type of movies/series and country

clean_df.groupBy('type','country').count().orderBy(desc('count')).show()

+-------+--------------+-----+
|   type|       country|count|
+-------+--------------+-----+
|  Movie| United States| 2058|
|  Movie|         India|  893|
|TV Show| United States|  760|
|  Movie|       Unknown|  440|
|TV Show|       Unknown|  391|
|TV Show|United Kingdom|  213|
|  Movie|United Kingdom|  206|
|TV Show|         Japan|  169|
|TV Show|   South Korea|  158|
|  Movie|        Canada|  122|
|  Movie|         Spain|   97|
|  Movie|         Egypt|   92|
|  Movie|       Nigeria|   86|
|TV Show|         India|   79|
|  Movie|     Indonesia|   77|
|  Movie|        Turkey|   76|
|  Movie|         Japan|   76|
|  Movie|        France|   75|
|  Movie|   Philippines|   73|
|  Movie|        Mexico|   70|
+-------+--------------+-----+
only showing top 20 rows

